# Fetching the data

The data that is used in the paper is related to various financial instruments:

- GOLD --> Gold futures
- Silver --> Silver futures
- Wheat -->	Wheat futures
- GAZ -->	Natural gas futures
- SP500 -->	S&P 500	
- NSDQ -->	Nasdaq Composite
- VIX -->	Volatility index
- IBEX -->	IBEX 35 (Spain)
- AEX -->	AEX (Netherlands)
- BEL -->	BEL 20 (Belgium)
- DAX -->	DAX (Germany)
- FTSE -->	FTSE 100 (UK)
- SMI -->	Swiss Market Index
- ASX -->	ASX 200 (Australia)
- N225 -->	Nikkei 225
- HIS -->	Hang Seng
- Nifty -->	Nifty 50 (India)
- BVSP -->	Bovespa (Brazil)	
- NZX -->	NZX 50 (New Zealand)
- US yield  -->	US 10-year Treasury

In [1]:
%pip install pandas requests yfinance

Note: you may need to restart the kernel to use updated packages.


In [2]:
import io
from pathlib import Path
 
import pandas as pd
import requests
import yfinance as yf

In [3]:
START = "2011-02-01"
END = "2026-02-28" 
PERIOD = '1d'

OUT = Path("data/raw")
OUT.mkdir(parents=True, exist_ok=True)

YAHOO = {
    "gold":   ["GC=F"],
    "silver": ["SI=F"],
    "wheat":  ["ZW=F"],
    "gaz":    ["NG=F"],
    "sp500":  ["^GSPC"],
    "nsdq":   ["^IXIC"],
    "vix":    ["^VIX"],
    "ibex":   ["^IBEX"],
    "aex":    ["^AEX"],
    "bel":    ["^BFX"],
    "dax":    ["^GDAXI"],
    "ftse":   ["^FTSE"],
    "smi":    ["^SSMI"],
    "asx":    ["^AXJO"],
    "n225":   ["^N225"],
    "his":    ["^HSI"],
    "nifty":  ["^NSEI"],
    "bvsp":   ["^BVSP"],
    "nzx":    ["^NZ50"],
    "usyield": ["^TNX"],
    "ta35":   ["TA35.TA"],
    "oil": ["CL=F"]
}

In [4]:
def fetch_single_asset(name):
    ticker = YAHOO[name]

    try:
        data = yf.download(
            tickers=ticker,
            interval=PERIOD,
            start=START,
            end=END
        )

        close = data["Close"].iloc[:, 0]
        
        close.name = name
        return close

    except Exception as e:
        print(f"{name}: {e}")
        return None


def fetch_all():

    # Fetch Yahoo data
    dico = {}

    for name in YAHOO.keys():
        result = fetch_single_asset(name)

        if result is not None:
            dico[name] = result

    yahoo_df = pd.concat(dico, axis=1, sort=True)

    # Fetch US yield data
    url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=DGS10"
    r = requests.get(url, timeout=30)
    r.raise_for_status()

    df = pd.read_csv(
        io.StringIO(r.text),
        index_col=0,
        parse_dates=True
    )

    usyield = pd.to_numeric(df.iloc[:, 0], errors="coerce").loc[START:END]
    usyield.name = "us_yield"

    # Fetch Japan 10Y yield
    url = "https://www.mof.go.jp/english/policy/jgbs/reference/interest_rate/historical/jgbcme_all.csv"
    r = requests.get(url, timeout=30)
    r.raise_for_status()

    df = pd.read_csv(url, skiprows=1, na_values=["-"])
    df["Date"] = pd.to_datetime(df["Date"])

    jpyield = pd.to_numeric(
        df.set_index("Date")["10Y"],
        errors="coerce"
    ).loc[START:END]

    jpyield.name = "jp_10yield"

    # Combine all data
    data = pd.concat(
        [yahoo_df, usyield, jpyield],
        axis=1,
        sort=True
    )

    return data

In [5]:
allData = fetch_all()

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%********

In [6]:
allData

,gold,silver,wheat,gaz,sp500,nsdq,vix,ibex,aex,bel,...,n225,his,nifty,bvsp,nzx,usyield,ta35,oil,us_yield,jp_10yield
2011-02-01,1339.599976,28.524000,835.75,4.347,1307.589966,2751.189941,17.629999,10967.889648,367.109985,2683.729980,...,10274.500000,23482.949219,5417.200195,67847.0,3348.459961,3.443,1289.709961,90.769997,3.48,1.237
2011-02-02,1331.500000,28.299000,863.00,4.429,1304.030029,2749.560059,17.299999,11010.189453,367.390015,2688.780029,...,10457.360352,23908.960938,5432.000000,66688.0,3352.300049,3.489,1300.680054,90.860001,3.52,1.231
2011-02-03,1352.300049,28.733000,859.00,4.337,1307.099976,2753.879883,16.690001,10860.689453,364.890015,2684.899902,...,10431.360352,NaN,5526.750000,66765.0,3349.889893,3.541,1299.119995,90.540001,3.58,1.244
2011-02-04,1348.300049,29.063999,853.75,4.310,1310.869995,2769.300049,15.930000,10854.689453,365.510010,2692.360107,...,10543.519531,NaN,5395.750000,65269.0,3367.820068,3.652,NaN,89.029999,3.68,1.288
2011-02-07,1347.599976,29.348000,858.75,4.104,1319.050049,2783.989990,16.280001,10922.588867,369.850006,2724.469971,...,10592.040039,23553.589844,5396.000000,65362.0,3387.409912,3.644,1323.680054,87.480003,3.68,1.292
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-02-23,5225.600098,86.523003,569.50,2.985,6837.750000,22627.269531,21.010000,18288.699219,1016.450012,5605.189941,...,NaN,27081.910156,25713.000000,188854.0,13382.709961,4.029,4243.470215,66.309998,4.03,NaN
2026-02-24,5176.299805,87.457001,567.50,2.915,6890.069824,22863.679688,19.549999,18189.500000,1019.869995,5629.149902,...,57321.089844,26590.320312,25424.650391,191490.0,13556.849609,4.033,4169.000000,65.629997,4.04,2.119
2026-02-25,5226.200195,90.939003,565.75,2.969,6946.129883,23152.080078,17.930000,18461.000000,1030.060059,5616.479980,...,58583.121094,26765.720703,25482.500000,191248.0,13517.480469,4.048,4162.899902,65.419998,4.05,2.154
2026-02-26,5194.200195,86.998001,571.75,2.827,6908.859863,22878.380859,18.629999,18496.599609,1022.400024,5439.049805,...,58753.390625,26381.019531,25496.550781,191005.0,13646.780273,4.017,4101.600098,65.209999,4.02,2.168
